### Convert Keras Models to ONNX

The below cell converts every `.keras` model in the `models/` directory to ONNX format, saving each `.ONNX` file in the same folder with the original filename. It automatically derives the input signature from each model, uses ONNX opset 13, and validates each converted model. Conversion errors are reported individually so the remaining models can still be processed. Original Keras files are preserved.

In [3]:
from pathlib import Path

import tensorflow as tf
import onnx
import tf2onnx


MODELS_DIR = Path("models")
OPSET = 17 # Matches the torch script, if changed here, please change there too

keras_files = sorted(MODELS_DIR.glob("*.keras"))

if not keras_files:
    print(f"No .keras models found in {MODELS_DIR.resolve()}")

for model_path in keras_files:
    onnx_path = model_path.with_suffix(".onnx")

    print(f"\nConverting: {model_path.name}")

    try:
        model = tf.keras.models.load_model(
            model_path,
            compile=False,
        )

        # Build an input signature from the model's actual input shape.
        input_signature = [
            tf.TensorSpec(
                shape=tuple(
                    None if dim is None else int(dim)
                    for dim in tensor.shape
                ),
                dtype=tensor.dtype,
                name=f"input_{i}",
            )
            for i, tensor in enumerate(model.inputs)
        ]

        @tf.function(input_signature=input_signature)
        def serving_fn(*inputs):
            result = model(
                inputs[0] if len(inputs) == 1 else list(inputs),
                training=False,
            )
            return result

        onnx_model, _ = tf2onnx.convert.from_function(
            serving_fn,
            input_signature=input_signature,
            opset=OPSET,
            output_path=str(onnx_path),
        )

        # Validate the generated ONNX model.
        onnx.checker.check_model(onnx_model)

        print(f"SUCCESS: {onnx_path.name}")

    except Exception as exc:
        print(f"FAILED: {model_path.name}")
        print(f"{type(exc).__name__}: {exc}")

print("\nConversion run complete.")



Converting: ai_capstone_keras_best_small_model.keras


I0000 00:00:1791886917.437978     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886917.438977     126 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-10-13 10:21:57.439061: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
I0000 00:00:1791886917.439251     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886917.440129     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so

SUCCESS: ai_capstone_keras_best_small_model.onnx

Converting: ai_capstone_keras_medium_model.keras


I0000 00:00:1791886917.812798     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886917.813782     126 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-10-13 10:21:57.813849: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
I0000 00:00:1791886917.814072     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886917.814942     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so

SUCCESS: ai_capstone_keras_medium_model.onnx

Converting: ai_capstone_keras_pure_vit_model.keras


I0000 00:00:1791886918.374222     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886918.375184     126 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-10-13 10:21:58.375251: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
I0000 00:00:1791886918.375436     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1791886918.376283     126 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so

SUCCESS: ai_capstone_keras_pure_vit_model.onnx

Conversion run complete.
